# WikiText-2 perplexity + benchmark FP16 × INT8 × INT4 (Colab)

Repo **privado** (`jeanjonatas/tcc-ptq-ppl`).

- **8B (Llama 3 / Qwen3):** Runtime → **A100** ou **L4** (Colab Pro). O FP16 do 8B ocupa ~15 GiB e **não cabe no T4**. INT4 (AWQ/GPTQ via `gptqmodel`) exige CUDA compute capability ≥ 8.0.
- **T4 (grátis):** só o perfil de validação com `Qwen/Qwen2.5-1.5B` (INT4 funciona nesse tamanho).

1. Colab → ícone de chave → secrets `GITHUB_TOKEN` (PAT com scope `repo`) e `HF_TOKEN` (token *Read* do Hugging Face, com acesso ao Llama 3 aprovado). Ligue *Notebook access* nos dois.
2. Rode as células em ordem. As do meio (dry-run, smoke PPL, pergunta) são opcionais.
3. O benchmark roda **um modelo por vez** e copia resultados + artefatos quantizados para o Drive depois de cada um, para não perder a calibração INT4 se a sessão cair.

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
from google.colab import userdata

REPO = "jeanjonatas/tcc-ptq-ppl"
REPO_DIR = Path("/content") / REPO.split("/")[-1]
token = userdata.get("GITHUB_TOKEN")

if REPO_DIR.joinpath("scripts/run_perplexity.py").exists():
    %cd {REPO_DIR}
    !git pull --ff-only
else:
    if not token:
        raise SystemExit("Crie o secret GITHUB_TOKEN no Colab (PAT com scope repo).")
    !git clone https://{token}@github.com/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
!git log --oneline -1

Opcional: cache do Hugging Face no Drive, para não baixar os modelos de novo se a sessão cair. Os dois 8B ocupam ~31 GB: só ligue se o Drive tiver espaço.

In [ ]:
USE_DRIVE_CACHE = False

if USE_DRIVE_CACHE:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    cache = "/content/drive/MyDrive/hf_cache"
    os.makedirs(cache, exist_ok=True)
    os.environ["HF_HOME"] = cache
    print("HF_HOME=", cache)

In [ ]:
import os
from pathlib import Path

if not Path("scripts/run_perplexity.py").exists():
    for candidate in Path("/content").glob("*/scripts/run_perplexity.py"):
        os.chdir(candidate.parents[1])
        break
print("cwd=", Path.cwd())
!pip install -q -r requirements-ppl.txt
# ninja é necessário para o gptqmodel compilar sob demanda a extensão C++ do
# kernel AWQ. Sem ninja o Colab quebra com "Ninja is required to load C++
# extensions". Instalado com deps porque é um pacote Python leve.
!pip install -q ninja
# gptqmodel sem recomputar deps: evita cataclismo de upgrades (torch/numpy/scipy)
# que quebra o ABI do stack pré-instalado do Colab.
!pip install -q --no-deps gptqmodel device-smi logbar tokenicer defuser pypcre
# Rede de segurança: se por qualquer motivo o torch do Colab tiver sido atualizado
# em uma rodada anterior, alinhar torch/torchvision/torchaudio juntos previne
# erros como "torchvision::nms does not exist" ou "TorchAudio compiled with
# different CUDA version" ao importar transformers.
import torch as _t
_cu_major = int((_t.version.cuda or "0.0").split(".")[0])
import importlib.metadata as _md
try:
    _tv_cu = _md.version("torchvision").split("+cu")[-1][:2]
except _md.PackageNotFoundError:
    _tv_cu = ""
if str(_cu_major) not in _tv_cu:
    print(f"torch cu={_t.version.cuda} vs torchvision cu={_tv_cu} — realinhando trio...")
    get_ipython().system("pip install -q --upgrade torch torchvision torchaudio")

print("\nATENÇÃO: reinicie o runtime agora (Runtime → Restart runtime) antes das próximas células,")
print("para o Python recarregar módulos nativos recém-instalados.")

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## Dry-run

Só dataset + detokenizer. Esperado: `n_pages_in_split` na casa de ~60, não 4358.

In [ ]:
!python scripts/run_perplexity.py --dry-run --output results/dryrun_colab.json

## Smoke PPL (4 docs)

Primeira vez baixa `Qwen/Qwen2.5-1.5B` (~3 GB). 2–5 min no T4 depois do download.

In [ ]:
!python scripts/run_perplexity.py \
  --model Qwen/Qwen2.5-1.5B \
  --max-docs 4 \
  --max-length 512 \
  --device cuda \
  --output results/ppl_smoke_colab.json

## Pergunta (smoke qualitativo, não é métrica oficial)

Modelo **base** completa texto. Para chat: troque para `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
from ppl.generate import generate_once, load_for_generate

PERGUNTA = "Explique em uma frase o que é perplexidade em modelos de linguagem."
model, tokenizer, device = load_for_generate("Qwen/Qwen2.5-1.5B", "cuda")
print(generate_once(model=model, tokenizer=tokenizer, prompt=PERGUNTA, max_new_tokens=64, device=device))

## Benchmark FP16 × INT8 × INT4 AWQ × INT4 GPTQ

Mesma tabela da bancada de referência: PPL, disco, pico de memória, TTFT e tokens/s, com variação e limiar de 3% por precisão.

A célula de configuração escolhe sozinha pela memória da GPU:

- **≥ 22 GiB (A100, L4):** `meta-llama/Meta-Llama-3-8B` e `Qwen/Qwen3-8B-Base`, perfil `full` (split completo, contexto 2048, 10 prompts × 10 reps, FP16 + INT8 + INT4-AWQ + INT4-GPTQ).
- **< 22 GiB (T4):** `Qwen/Qwen2.5-1.5B`, perfil `colab` (valida o caminho CUDA; INT4 funciona nessa GPU para 1.5B).

Tempo estimado no A100: ~1h por modelo 8B (download + FP16 + INT8 + calibração C4 × 2 + INT4-AWQ + INT4-GPTQ + auditoria GPTQ). A calibração INT4 reaproveita o diretório `artifacts/<modelo>-int4-*/` se já existir (passe `--requantize` para refazer).

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN carregado do secret")
except Exception:
    print("sem secret HF_TOKEN: ok para Qwen, necessário para o Llama 3")

In [ ]:
import torch
from datetime import datetime, timezone

gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if gpu_gib >= 22:
    PROFILE = "full"
    MODELS = ["meta-llama/Meta-Llama-3-8B", "Qwen/Qwen3-8B-Base"]
else:
    PROFILE = "colab"
    MODELS = ["Qwen/Qwen2.5-1.5B"]

# Para forçar outra escolha, edite aqui, por exemplo:
# PROFILE, MODELS = "full", ["Qwen/Qwen3-8B-Base"]

SAVE_TO_DRIVE = True  # copia results/ para o Drive depois de cada modelo
RUN_DIR = f"results/{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{PROFILE}_colab"
print(f"GPU {torch.cuda.get_device_name(0)} ({gpu_gib:.1f} GiB) -> PROFILE={PROFILE} MODELS={MODELS}")
MODELS_CSV = ",".join(MODELS)
print("RUN_DIR =", RUN_DIR)

In [ ]:
!python docker/selfcheck.py
!python scripts/check_access.py --models {MODELS_CSV}

### Smoke INT4 (valida antes de queimar 2h no 8B)

Dois fixes recentes do pipeline são o que decide se o INT4 vai concluir:

- `_load_gptqmodel` passa `device_map="auto"` (único aceito pelo gptqmodel 7.x).
- `ninja` instalado para a extensão JIT do kernel AWQ compilar.

A célula abaixo exercita os dois caminhos em ~15 min, antes de entrar no benchmark 8B:

- **GPTQ load**: reaproveita um checkpoint `*-int4-gptq` já salvo no Drive (sem recalibrar). Valida só o load.
- **AWQ quantize**: calibra `Qwen/Qwen2.5-1.5B` com 16 amostras do C4 (modelo pequeno, calibração abreviada). Força o gptqmodel a compilar o kernel AWQ via `ninja`.

Se qualquer parte falhar, **pare e investigue** — o benchmark 8B vai cair no mesmo lugar depois de ~2h.

In [ ]:
import shutil, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
Path("artifacts").mkdir(exist_ok=True)

# monta o Drive se ainda não foi montado (idempotente)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as _e:
    print(f"[smoke] aviso: Drive não montado ({_e!r}) — smoke do GPTQ load será pulado")

# --- smoke 1: INT4-GPTQ load (valida fix do device_map) ---
drive_art = Path("/content/drive/MyDrive/tcc-ptq-artifacts")
candidates = sorted(drive_art.glob("*int4-gptq")) if drive_art.exists() else []
if not candidates:
    print("[smoke-gptq] pulado: nenhum artefato *-int4-gptq no Drive ainda. Será exercitado no primeiro benchmark 8B.")
else:
    src = candidates[0]
    local = Path("artifacts") / src.name
    if not local.exists():
        print(f"[smoke-gptq] copiando {src.name} do Drive -> artifacts/ ...")
        shutil.copytree(src, local)
    print(f"[smoke-gptq] carregando {local} com precision=int4-gptq ...")
    from ppl.loading import load_model
    _m = load_model(str(local), precision="int4-gptq", device="cuda")
    print(f"[smoke-gptq] OK — modelo carregou como {type(_m).__name__}")
    del _m
    import torch
    torch.cuda.empty_cache()

# --- smoke 2: INT4-AWQ quantize (valida fix do ninja + compile JIT) ---
# Qwen2.5-1.5B + nsamples=16 é só para forçar o kernel AWQ a compilar.
# NÃO use este checkpoint em medidas: a calibração é truncada de propósito.
print("[smoke-awq] quantizando Qwen/Qwen2.5-1.5B (nsamples=16) para forçar o compile JIT ...")
from ppl.quantize import quantize_awq
quantize_awq("Qwen/Qwen2.5-1.5B", "artifacts/smoke-awq-1.5b", nsamples=16)
print("[smoke-awq] OK — AWQ compilou e quantizou")

print("\n✅ smoke INT4 completo — pode seguir para o benchmark 8B")

In [ ]:
import shutil
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/tcc-ptq-results")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    ART_DRIVE = Path("/content/drive/MyDrive/tcc-ptq-artifacts")
    ART_DRIVE.mkdir(parents=True, exist_ok=True)

for model in MODELS:
    print(f"\n========== {model} ==========")
    !python scripts/run_benchmark.py --profile {PROFILE} --models {model} --run-dir {RUN_DIR}
    if SAVE_TO_DRIVE:
        shutil.copytree(RUN_DIR, DRIVE_DIR / Path(RUN_DIR).name, dirs_exist_ok=True)
        print("resultados copiados para", DRIVE_DIR / Path(RUN_DIR).name)
        # salva também os pesos quantizados (AWQ/GPTQ) para evitar recalibrar se a sessão cair
        for art in Path("artifacts").glob("*-int4-*"):
            dst = ART_DRIVE / art.name
            shutil.copytree(art, dst, dirs_exist_ok=True)
            print("artefato copiado para", dst)

A tabela aparece no fim do log acima e em `RUN_DIR/summary.md`. Se a sessão caiu no meio, rode de novo só o modelo que faltou com o mesmo `RUN_DIR` (o `summary.md` junta todos os JSONs da pasta). Para mesclar pastas diferentes: `python scripts/compare.py --run-dir A --run-dir B --out results/merged`.

In [ ]:
from IPython.display import Markdown, display

print(RUN_DIR)
display(Markdown(Path(RUN_DIR, "summary.md").read_text()))

In [ ]:
!zip -qr results_colab.zip {RUN_DIR} logs
from google.colab import files
files.download("results_colab.zip")

## Opcional: cruzar com `lm_eval --limit 4`

`word_perplexity` deve ficar na mesma casa do JSON do smoke (diferença só de arredondamento).

In [ ]:
RUN_LM_EVAL = False

if RUN_LM_EVAL:
    !pip install -q "lm-eval[hf]==0.4.13"
    !lm_eval --model hf \
      --model_args pretrained=Qwen/Qwen2.5-1.5B,max_length=512 \
      --tasks wikitext --limit 4 --device cuda

Baixe os arquivos em `results/` e `logs/` pelo painel de arquivos do Colab.